# Module 36 — Loop Engineering: Production Execution Kernel
**Goal:** engineer the control loop before engineering the reusable harness.
Practice: Predict → Build → Try → Observe → Break → Debug → Measure → Improve → Defend.
This API-key-free notebook uses deterministic simulators. Model output is an untrusted proposal; policy, budgets, execution and verification remain outside it.

## Architecture
```text
Task Contract → OBSERVE → DECIDE/PROPOSE → POLICY → ACT → VERIFY
                    ↑                                  ↓
                    └──────── RECOVER / REPLAN ─────────┘
                              ↓
                    COMPLETE / STOP / ESCALATE
```
Control plane: identity, tenant, policy, approval, budget, deadline, kill switch.
Data plane: user text, retrieval, memory, tool output, external observations.
**Module 36 defines semantics; Module 37 packages them into a reusable harness.**

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
from importlib import import_module
m = import_module("36-loop-engineering.app.loop_engine")
LoopEngine, LoopState = m.LoopEngine, m.LoopState
Task, Observation, Proposal = m.Task, m.Observation, m.Proposal
Phase, Outcome = m.Phase, m.Outcome
stable_action_id, stop_if_budget_exhausted = m.stable_action_id, m.stop_if_budget_exhausted
print("Loaded:", LoopEngine.__name__)

## Lab 1 — State-machine contract
Define valid transitions and their preconditions.
- OBSERVE → DECIDE: fresh, in-scope evidence
- DECIDE → POLICY: typed proposal
- POLICY → ACT: authorization + budget + approval
- ACT → VERIFY: recorded attempt
- VERIFY → COMPLETE: independent postcondition
- UNKNOWN → RETRY: reconciliation/idempotency proof
Challenge: document one invalid transition and its rejection behavior.

In [ ]:
task = Task("soc-001", "tenant-a", "investigate login", max_steps=4)
proposal = Proposal("disable_account", {"tenant_id":"tenant-b","user_id":"u7"})
print(proposal)
print("action_id:", stable_action_id(task.task_id, proposal))
print("Expected: DENY — model output cannot redefine tenant scope.")

## Lab 2 — Observation freshness and provenance
For each observation capture source, timestamp/freshness, scope, provenance, payload and state version.
**Break:** return stale evidence. Reacquire or escalate; never treat stale evidence as current truth.

In [ ]:
def observer(state):
    return Observation(f"obs-{state.step}", "SIEM",
                       {"tenant_id":state.task.tenant_id,"risk":"high"},
                       state.state_version+1, fresh=True)
def decider(state, obs):
    return Proposal("read_events", {"tenant_id":state.task.tenant_id,"window_minutes":30})
def policy(state, p):
    return p.arguments.get("tenant_id")==state.task.tenant_id, "tenant_scope"
def actor(state, p, aid):
    return {"status":"ok","action_id":aid,"events":3}
def verifier(state, p, result):
    return result.get("status")=="ok" and result.get("events",0)>0
out = LoopEngine(observer,decider,policy,actor,verifier).run(
    LoopState(Task("soc-002","tenant-a","investigate",max_steps=3)))
print(out.result, out.phase, "trace:", len(out.trace))

## Lab 3 — Policy and authority attacks
Try: wrong tenant, fabricated approval, security-control bypass, budget increase in retrieved text, and a poisoned tool message.
**Gold property:** untrusted data cannot rewrite identity, tenant, policy, approval or budget.

In [ ]:
attacks = [
 Proposal("read_events",{"tenant_id":"tenant-b"}),
 Proposal("disable_account",{"tenant_id":"tenant-a","approval_id":"fake"}),
 Proposal("export_all",{"tenant_id":"tenant-a"})]
for p in attacks:
    allowed, reason = policy(out,p)
    print(p.action, "ALLOW" if allowed else "DENY", reason)

## Lab 4 — Retry taxonomy
Classify before retrying: timeout/rate-limit → bounded retry; malformed arguments → repair/reject; authorization/policy denial → stop/escalate; stale evidence → reobserve; unknown side effect → reconcile; dependency outage → circuit-break/safe-stop.
Deliverable: a 10-row failure decision table.

In [ ]:
retry_matrix = {
 "timeout":"RETRY","rate_limit":"RETRY","malformed_args":"REPAIR_OR_STOP",
 "authorization_denied":"ESCALATE","policy_denied":"ESCALATE",
 "stale_observation":"REOBSERVE","unknown_side_effect":"RECONCILE",
 "dependency_outage":"SAFE_STOP"}
for k,v in retry_matrix.items(): print(f"{k:24} -> {v}")

## Lab 5 — Side-effect safety
`write → timeout` means **UNKNOWN**, not necessarily failure. Reconcile external state using action identity. Retry only when safe.

In [ ]:
p=Proposal("issue_refund",{"order_id":"o-17","amount":75,"currency":"USD"})
aid=stable_action_id("refund-1",p); ledger={}
def write_once(action_id, amount):
    if action_id in ledger: return {"status":"deduplicated","amount":ledger[action_id]}
    ledger[action_id]=amount; return {"status":"applied","amount":amount}
print(write_once(aid,75)); print(write_once(aid,75)); assert ledger[aid]==75

## Lab 6 — Independent verification
Compare L0 self-report, L1 same-source confirmation, L2 independent query, L3 invariant/verifier, L4 human confirmation.
**Break:** model says rollback succeeded while external state is still wrong. Do not complete.

In [ ]:
external={"version":"v1","error_rate":0.12}
claim={"status":"rollback complete"}
print("claim:",claim,"external:",external)
assert external["version"]!="target-v0"
print("Correct: verification fails.")

## Lab 7 — Loop pathology
Create fixtures for A↔B oscillation, repeated observations, repeated actions, search↔summarize ping-pong and plan churn. Combine hard budgets with progress/repetition signals.

In [ ]:
def fingerprint(obs,p): return (repr(sorted(obs.payload.items())),p.action,repr(sorted(p.arguments.items())))
obs=Observation("o","fake",{"same":True},1)
p=Proposal("read",{"resource":"r"})
print(fingerprint(obs,p))
print("Use repeated fingerprints as a signal, not the sole termination rule.")

## Lab 8 — Budget engineering
Track steps, model calls, tool calls, tokens, cost, wall time and high-risk actions. The model may observe remaining budget but must never increase it.

In [ ]:
s=LoopState(Task("budget-1","tenant-a","bounded",max_steps=3,max_model_calls=3,max_tool_calls=2,max_cost=.10),
              step=3,model_calls=3,tool_calls=2,cost=.10)
print("Budget decision:",stop_if_budget_exhausted(s))

## Lab 9 — Checkpoint/recovery
Checkpoint semantic state, not just a loop counter. Preserve task/tenant, state version, approved action, committed side effects, verification, remaining budget and recovery position.
Crash at: post-observation, pre-side-effect, post-side-effect, post-verification. Prove no duplicate side effect.

In [ ]:
checkpoint={"task_id":"task-9","state_version":7,"phase":"ACT",
"approved_action_id":aid,"completed_action_ids":[],"verification":None,
"budget_remaining":{"tool_calls":4,"cost":.40}}
print(checkpoint)

## Lab 10 — Concurrency and stale commits
A branch result at state version 5 must not overwrite authoritative state already committed at version 6. Design optimistic concurrency, leasing, or serialization.

In [ ]:
authoritative_version=6; branch_version=5
can_commit=branch_version>=authoritative_version
print("stale branch may commit:",can_commit); assert not can_commit

## Lab 11 — Cancellation and deadlines
Cancellation is a control event: reject new work, cancel interruptible work, block new side effects, record cause, and preserve safe resume state when permitted.
Exercise: propagate a 120-second deadline into model/retrieval/tool timeouts.

## Lab 12 — Replayable trajectory
Record run_id, task_id, tenant, state_version, phase, observation_ref, proposal, policy decision, action_id, verifier result, error class, latency, cost and stop reason. Replay decision-relevant behavior and external effects; do not pretend model tokens are perfectly deterministic.

In [ ]:
trajectory=[
 {"phase":"OBSERVE","state_version":1,"observation_ref":"obs-1"},
 {"phase":"DECIDE","proposal":"read_events"},
 {"phase":"POLICY","allowed":True},
 {"phase":"ACT","action_id":aid},
 {"phase":"VERIFY","ok":True}]
for e in trajectory: print(e)

## Lab 13 — Differential benchmark
Run the same fixtures through fixed, reactive and guarded-adaptive loops. Compare verified success, unsafe acceptance, recovery, tool calls/success, p95 latency, cost/success, escalation and loop pathology. Never assume more autonomy is better.

In [ ]:
rows=[
 {"architecture":"fixed","success":.80,"safety":.99,"cost":.20},
 {"architecture":"reactive","success":.86,"safety":.93,"cost":.42},
 {"architecture":"guarded_adaptive","success":.88,"safety":.995,"cost":.31}]
for r in rows: print(r)

## Lab 14 — Failure-injection matrix
Inject provider outage, stale observation, timeout, rate limit, duplicate delivery, malformed action, policy/authorization denial, budget exhaustion, deadline expiry, false verifier, worker crash, corrupted checkpoint, prompt injection, poisoned tool output and unknown side effect.
For each: detection → containment → retry/recover/stop → expected result → residual risk → regression test.

## Lab 15 — Production design challenge
Design an autonomous SOC investigator for **10,000 alerts/day**. Defend state transitions, evidence freshness, typed proposals, identity/tenant boundary, approvals, idempotency, verification, retries, budgets, cancellation, concurrency, checkpointing, replay, audit and evaluation.
**Mastery:** survive malicious proposal, stale evidence, transient failure, unknown side effect, false verifier, worker crash and exhausted budget without losing control.

## Handoff — Loop Contract v1
Submit: state model; transition table; action schema; policy insertion points; verification/postconditions; retry taxonomy; action identity/idempotency; budget/deadline rules; stop/escalation rules; checkpoint boundaries; trajectory schema; failure matrix; benchmark fixtures.
**Module 36 = engineer the loop. Module 37 = engineer the reusable harness around it.**